In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("../data/medical_symptoms_dataset.csv")
data.head()

,age,gender,fever,cough,fatigue,headache,muscle_pain,nausea,vomiting,diarrhea,...,diastolic_bp,heart_rate,temperature_c,oxygen_saturation,wbc_count,hemoglobin,platelet_count,crp_level,glucose_level,diagnosis
0,52,Female,1,0,2,0,0,3,0,3,...,68.0,79.0,36.2,99.0,5.32,13.47,335.0,3.95,119.0,Dengue
1,15,Male,0,0,1,0,1,0,1,0,...,72.0,79.0,37.6,97.0,10.31,14.73,273.0,18.03,104.0,Dengue
2,72,Female,0,2,2,0,2,3,1,1,...,76.0,70.0,37.1,94.0,7.03,11.10,307.0,22.77,120.0,Influenza
3,61,Female,2,0,0,1,0,1,2,1,...,67.0,98.0,36.2,96.0,7.23,12.55,221.0,5.83,97.0,COVID-19
4,21,Female,0,0,0,0,2,3,1,1,...,77.0,103.0,35.0,96.0,9.37,12.69,280.0,16.14,128.0,Dengue


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                3000 non-null   int64  
 1   gender             3000 non-null   str    
 2   fever              3000 non-null   int64  
 3   cough              3000 non-null   int64  
 4   fatigue            3000 non-null   int64  
 5   headache           3000 non-null   int64  
 6   muscle_pain        3000 non-null   int64  
 7   nausea             3000 non-null   int64  
 8   vomiting           3000 non-null   int64  
 9   diarrhea           3000 non-null   int64  
 10  skin_rash          3000 non-null   int64  
 11  loss_smell         3000 non-null   int64  
 12  loss_taste         3000 non-null   int64  
 13  systolic_bp        3000 non-null   float64
 14  diastolic_bp       3000 non-null   float64
 15  heart_rate         3000 non-null   float64
 16  temperature_c      3000 non-null   

In [4]:
print(data["diagnosis"].nunique())
print(data["diagnosis"].unique().tolist())
print(data["gender"].nunique())
print(data["gender"].unique().tolist())

5
['Dengue', 'Influenza', 'COVID-19', 'Malaria', 'Pneumonia']
2
['Female', 'Male']


In [5]:
data["gender"] = data["gender"].map({"Male":1,"Female":0})
data["diagnosis"] = data["diagnosis"].map({
    "Dengue":0,
    "Influenza":1,
    "COVID-19":2,
    "Malaria":3,
    "Pneumonia":4
})
data.head()

,age,gender,fever,cough,fatigue,headache,muscle_pain,nausea,vomiting,diarrhea,...,diastolic_bp,heart_rate,temperature_c,oxygen_saturation,wbc_count,hemoglobin,platelet_count,crp_level,glucose_level,diagnosis
0,52,0,1,0,2,0,0,3,0,3,...,68.0,79.0,36.2,99.0,5.32,13.47,335.0,3.95,119.0,0
1,15,1,0,0,1,0,1,0,1,0,...,72.0,79.0,37.6,97.0,10.31,14.73,273.0,18.03,104.0,0
2,72,0,0,2,2,0,2,3,1,1,...,76.0,70.0,37.1,94.0,7.03,11.10,307.0,22.77,120.0,1
3,61,0,2,0,0,1,0,1,2,1,...,67.0,98.0,36.2,96.0,7.23,12.55,221.0,5.83,97.0,2
4,21,0,0,0,0,0,2,3,1,1,...,77.0,103.0,35.0,96.0,9.37,12.69,280.0,16.14,128.0,0


In [6]:
x = data.drop(columns=["diagnosis"])
y = data["diagnosis"]
print(data["diagnosis"].value_counts())

diagnosis
1    748
2    711
0    639
3    464
4    438
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(2400, 23)
(600, 23)
(2400,)
(600,)


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer

num_features = x.select_dtypes(include=["int64","float64"]).columns
cat_features = x.select_dtypes(include=["object"]).columns

num_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="mean")),
    ("scaler",StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num",num_pipeline,num_features),
    ("cat",cat_pipeline,cat_features)
])

In [9]:
x_train_preprocessed = preprocessor.fit_transform(x_train)
x_test_preprocessed = preprocessor.transform(x_test)
print(x_train_preprocessed.shape)
print(x_test_preprocessed.shape)

(2400, 23)
(600, 23)


In [10]:
import torch

x_train = torch.tensor(x_train_preprocessed, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.long)

x_test = torch.tensor(x_test_preprocessed, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.long)



In [11]:
print(x_train.mean(), x_train.std())

tensor(1.9004e-09) tensor(1.0000)


In [12]:
from torch.utils.data import TensorDataset,DataLoader

train_dataset = TensorDataset(x_train,y_train)
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)

In [13]:
import torch.nn as nn 

class MLP(nn.Module):
    def __init__(self,train_size):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(train_size,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,5),
            nn.Softmax()
        )
        
    def forward(self,x):
        return self.model(x)    

In [14]:
model = MLP(train_size=x_train.shape[1])

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.0005)

In [15]:
epochs = 401

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb,yb in train_loader:
        output = model(xb)
        loss = criterion(output,yb)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f} , avg_loss: {avg_loss:.4f}")

c:\Users\nigam\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\module.py:1779: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Epoch 0, Loss: 120.1804 , avg_loss: 1.6024
Epoch 50, Loss: 96.8369 , avg_loss: 1.2912
Epoch 100, Loss: 86.0456 , avg_loss: 1.1473
Epoch 150, Loss: 84.9519 , avg_loss: 1.1327
Epoch 200, Loss: 84.7888 , avg_loss: 1.1305
Epoch 250, Loss: 84.8072 , avg_loss: 1.1308
Epoch 300, Loss: 84.3899 , avg_loss: 1.1252
Epoch 350, Loss: 84.9662 , avg_loss: 1.1329
Epoch 400, Loss: 84.4995 , avg_loss: 1.1267


In [16]:
model.eval()

with torch.no_grad():
    pred = model(x_test)
    predicted = torch.argmax(pred,dim=1)
    accuracy = (predicted == y_test).float().mean()

print("Test Accuracy:", accuracy.item())    

Test Accuracy: 0.21166667342185974


In [17]:
with torch.no_grad():
    preds = torch.argmax(model(x_test), dim=1)

print(torch.bincount(preds))

tensor([128, 133, 137, 129,  73])


In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(x_train_preprocessed, y_train.numpy())

print("RF Accuracy:", rf.score(x_test_preprocessed, y_test.numpy()))

RF Accuracy: 0.21666666666666667


In [19]:
import pandas as pd

df = pd.DataFrame(x_train_preprocessed)
df["target"] = y_train.numpy()

print(df.groupby("target").mean().head())

               0         1         2         3         4         5         6  \
target                                                                         
0      -0.000103  0.024372  0.036721 -0.075030  0.047455 -0.038592 -0.031519   
1       0.067481  0.030501  0.014383 -0.045411 -0.056501  0.038402  0.050511   
2      -0.030614  0.004408 -0.055711  0.102272  0.010606  0.033190 -0.034654   
3      -0.056622 -0.036731  0.046804  0.011033  0.005464 -0.047383  0.090225   
4      -0.003217 -0.056209 -0.036063  0.006450  0.002146 -0.012779 -0.077586   

               7         8         9  ...        13        14        15  \
target                                ...                                 
0      -0.057778  0.014350 -0.004228  ... -0.049372 -0.010302  0.017334   
1      -0.049190 -0.047636 -0.047875  ...  0.037376  0.012566 -0.042620   
2       0.032006  0.047883  0.004173  ... -0.053817 -0.028313  0.047081   
3       0.033535  0.016351  0.016351  ...  0.078149  0.034412 -0